1. 정규표현식 함수와 re.sub의 동작 분석

Q1. findall과 캡처 그룹

In [76]:
import re

text = "강의는 2026-05-06, 숙제는 2026-05-18에 마감"

In [77]:
print(re.match(r"\d+", text))

None


(a)
결과: None
설명: re.match는 문자열의 맨 처음부터 매칭을 시도한다. text는 "강의는"으로 시작하므로 숫자 1개 이상을 의미하는 \d+와 매칭되지 않아 None을 반환되는 것을 알 수 있다.

In [78]:
print(re.search(r"\d{4}-\d{2}-\d{2}", text).group())

2026-05-06


(b)
결과: 2026-05-06
설명: re.search는 문자열 전체에서 패턴을 탐색해 첫 번째 매칭을 반환하는데, .group()는 매칭된 문자열을 반환하므로 첫 번째 날짜인 2026-05-06이 반환되는 것을 알 수 있다.

In [79]:
print(re.findall(r"\d{4}-\d{2}-\d{2}", text))

['2026-05-06', '2026-05-18']


(c)
결과: ['2026-05-06', '2026-05-18']
설명: re.findall은 매칭되는 모든 문자열을 리스트로 반환한다. 캡처 그룹이 없기 때문에 매칭 전체 문자열들이 반환되는 것을 알 수 있다.

In [80]:
re.findall(r"(\d{4})-(\d{2})-(\d{2})", text)

[('2026', '05', '06'), ('2026', '05', '18')]

(d)
결과: [('2026', '05', '06'), ('2026', '05', '18')]
설명: (c)와 다르게 캡처 그룹 ()이 3개 있으므로, findall은 각 매칭에서 그룹별로 묶인 튜플들의 리스트를 반환한다.

In [81]:
print(re.findall(r"(?:\d{4})-(?:\d{2})-(?:\d{2})", text))

['2026-05-06', '2026-05-18']


(e)
결과: ['2026-05-06', '2026-05-18']
설명: (?:)는 비캡처 그룹으로, 그룹핑은 하지만 캡처하지 않는다. 캡처 그룹이 없으므로 (c)와 동일하게 매칭 전체 문자열들의 리스트가 반환되는 것을 알 수 있다.

추가질문
설명: re.findall은 캡처 그룹 ()의 유무에 따라 반환 형태가 달라진다. (c)는 캡처 그룹이 없는 경우, (e)는 비캡처 그룹을 사용하는 경우이므로 매칭된 전체 문자열의 리스트를 반환하지만, (d)는 캡처 그룹이 3개 존재하므로 각 그룹에 해당하는 값을 튜플로 묶어 리스트로 반환한다.

Q2. 수량자와 re.sub 치환의 함정

In [82]:
import re

html = "<b>안녕</b>  <i>세상</i>!"
nums = "수강생 30명,  조교 3명"

In [83]:
print(re.sub(r"<.+>", "[T]", html))

[T]!


(a)
결과: [T]!
설명: .+는 탐욕적(greedy) 매칭으로 가능한 많이 매칭하므로 <b>안녕</b>  <i>세상</i> 전체를 하나로 매칭해 [T]로 치환한다.

In [84]:
print(re.sub(r"<.+?>", "[T]", html))

[T]안녕[T]  [T]세상[T]!


(b)
결과: [T]안녕[T]  [T]세상[T]!
설명: .+?는 게으른(lazy) 매칭으로 가능한 적게 매칭하므로 각 태그 <b>, </b>, <i>, </i>를 개별적으로 매칭해 각각 [T]로 치환한다.

In [85]:
print(re.sub(r"<[^>]+>", "[T]", html))

[T]안녕[T]  [T]세상[T]!


(c)
결과: [T]안녕[T]  [T]세상[T]!
설명: [^>]+는 > 이외의 문자를 1개 이상 매칭한다. >를 넘어가지 못하므로 (b)와 동일하게 각 태그를 개별 치환한다.

In [86]:
print(re.sub(r"(\d+)", r"<\1>", nums))

수강생 <30>명,  조교 <3>명


(d)
결과: 수강생 <30>명,  조교 <3>명
설명: r"<\1>"에서 \1은 첫 번째 캡처 그룹 (\d+)의 매칭값을 참조하는 역할을 하므로, 숫자 30과 3을 각각 <30>, <3>으로 치환한다.

In [87]:
print(re.sub(r"(\d+)", "<\1>", nums))

수강생 <>명,  조교 <>명


(e)
결과: 위 실행 결과
설명: "<\1>"에서 \1은 원시 문자열이 아니다. 따라서 \1은 백슬래시+숫자가 아닌 ASCII 제어문자로 해석되므로 숫자가 제어문자로 치환된다.

추가질문 1
설명: (a)에서 .+는 탐욕적 매칭으로 첫 <부터 마지막 >까지 전체를 하나로 매칭하지만, (b)에서 .+?는 게으른 매칭으로 <와 가장 가까운 > 사이만 매칭하므로 각 태그를 개별적으로 치환한다.

추가질문 2
설명: (d)에서는 r"<\1>"로 앞에 r을 붙여 원시 문자열을 사용해 \1이 첫 번째 캡처 그룹 참조로 해석되지만, (e)에서는 일반 문자열 "<\1>"을 사용해 \1이 ASCII 제어문자로 해석되므로 숫자가 올바르게 치환되지 않는다.

2. 한국어 소셜미디어 텍스트 정제 파이프라인

Q3. 강의 SNS 게시물 분석기

(a) clean_post(post: str)-> str

In [88]:
import re

# 문제에서 권장한 같은 패턴이 두 번 이상 사용될 시 re.compile로 미리 컴파일
URL_PAT     = re.compile(r"https?://\S+")          # http:// 또는 https://로 시작해 공백 전까지
HTML_PAT    = re.compile(r"<[^>]+>")               # < > 사이의 HTML 태그
EMAIL_PAT   = re.compile(r"\S+@\S+")               # 공백이 아닌 문자 + @ + 공백이 아닌 문자
PHONE_PAT   = re.compile(r"\d{2,4}-\d{3,4}-\d{4}") # 전화번호 형식 \d{2,4}-\d{3,4}-\d{4}
MENTION_PAT = re.compile(r"@\w+")                  # @로 시작하는 멘션
HASHTAG_PAT = re.compile(r"#\w+")                  # #으로 시작하는 해시태그
JAMO_PAT    = re.compile(r"[\u3131-\u3163]+")      # 유니코드 U+3131-U+3163 한글 자음/모음

def clean_post(post: str) -> str:
    post = URL_PAT.sub(" ", post)                 # 1단계: URL을 공백 한 칸으로 치환
    post = HTML_PAT.sub("", post)                 # 2단계: HTML 태그를 빈 문자열로 제거
    post, _ = EMAIL_PAT.subn("[이메일]", post)    # 3단계: 이메일을 [이메일]로 마스킹
    post, _ = PHONE_PAT.subn("[전화]", post)      # 3단계: 전화번호를 [전화]로 마스킹
    post = MENTION_PAT.sub(" ", post)             # 4단계: 멘션을 공백 한 칸으로 치환
    post = HASHTAG_PAT.sub(" ", post)             # 4단계: 해시태그를 공백 한 칸으로 치환
    post = JAMO_PAT.sub("", post)                 # 5단계: 한글 자음/모음을 빈 문자열로 제거
    post = re.sub(r"\s+", " ", post).strip()      # 6단계: 연속 공백을 한 칸으로 정리 후 앞뒤 공백 제거
    return post

(a)
설명: clean_post는 SNS 게시물 텍스트를 6단계 순서대로 정제하는 함수이다. 각 단계에서 반복 사용되는 패턴은 re.compile로 미리 컴파일하였다. 6단계의 순서를 반드시 지켜야 하는 이유는 예를 들어, 단계 3과 단계 4의 순서를 바꾸면 @prof_kim과 같은 멘션이 먼저 공백으로 치환된 후 이메일@도메인 형태의 이메일 패턴이 깨지면서 이메일이 올바르게 마스킹되지 않는 문제가 발생하기 때문이다.

(b) extract_hashtags(post: str)-> list[str]

In [89]:
HASHTAG_EXTRACT_PAT = re.compile(r"#([A-Za-z0-9\uAC00-\uD7A3]+)") # 원본 입력에서 해시태그를 추출하므로 별도 컴파일
                                                                  # #뒤에 한글(AC00-D7A3), 영문 대소문자, 숫자만 인정

def extract_hashtags(post: str) -> list[str]:
    return HASHTAG_EXTRACT_PAT.findall(post)  # 캡처 그룹으로 # 제외한 태그명만 반환

(b)
설명: extract_hashtags는 원본 게시물에서 # 뒤에 한글(AC00-D7A3), 영문 대소문자, 숫자로 이루어진 해시태그를 추출하는 함수다. 캡처 그룹 ()을 사용해 #은 제외하고 태그 이름만 리스트로 반환하며, 같은 태그가 여러 번 등장하면 중복 제거 없이 모두 포함한다.

(c) analyze_posts(posts: list[str])-> dict

In [90]:
from collections import Counter

def analyze_posts(posts: list[str]) -> dict:  # 모든 게시물을 clean_post로 정제
    cleaned = [clean_post(p) for p in posts]
    
    avg_length = round(sum(len(c) for c in cleaned) / len(cleaned), 2) # 정제된 게시물들의 평균 글자 수를 소수점 둘째 자리로 반올림
    
    all_tags = [] # 모든 원본 게시물에서 해시태그를 추출해 하나의 리스트로 합침
    for p in posts:
        all_tags.extend(extract_hashtags(p))  # 각 게시물의 해시태그를 누적
    
    hashtag_counts = dict(Counter(all_tags).most_common()) # Counter로 빈도 집계 후 most_common()으로 내림차순 정렬된 dict 생성
    
    masked_count = 0 # subn의 두 번째 반환값(치환 횟수)으로 마스킹된 개인정보 총 건수 집계
    for p in posts:
        _, n1 = EMAIL_PAT.subn("[이메일]", p)  # 이메일 치환 횟수
        _, n2 = PHONE_PAT.subn("[전화]", p)    # 전화번호 치환 횟수
        masked_count += n1 + n2                # 두 횟수를 합산

    return {
        "posts_n": len(posts),                        # 게시물 수
        "avg_length_after_clean": avg_length,         # 정제 후 평균 글자 수
        "hashtag_counts": hashtag_counts,             # 해시태그 빈도 (내림차순)
        "masked_count": masked_count,                 # 마스킹된 개인정보 총 건수
    }

(c)
설명: analyze_posts는 게시물 리스트를 받아 정제 후 평균 글자 수, 해시태그 빈도, 마스킹된 개인정보 건수를 딕셔너리로 반환하는 함수이다. 해시태그 빈도는 from collections import Counter과 most_common()을 활용해 빈도 내림차순으로 정렬했다. 마스킹 건수는 re.subn이 반환하는 치환 횟수를 직접 활용해 텍스트에서 [이메일], [전화]를 세는 방식 대신 정확한 치환 횟수를 집계했다.

In [91]:
posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",
    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",
    "<b>중요</b>: 다음 시험 범위는 1-15장. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",
    "  여러   공백과\n\n\n줄바꿈이   많은  텍스트  ",
    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

(1) 각 게시물에 대한 clean_post의 반환값 5개

In [92]:
for p in posts:
    print(clean_post(p))

print()

오늘 수업 진짜 재밌었음!! 감사 자료:
팀플 어디서 모이지 카톡
중요: 다음 시험 범위는 1-15장. 문의는 [이메일] ([전화])로!
여러 공백과 줄바꿈이 많은 텍스트
진짜 좋다



(2) analyze_posts(posts)의 반환 딕셔너리 전체

In [93]:
print(analyze_posts(posts))

{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


요약
설명: (a)에서 clean_post는 6단계를 순서대로 적용하는데 단계 순서가 중요한 이유는 예를 들어 단계 3과 단계 4의 순서를 바꾸면 멘션(@prof_kim)이 먼저 공백으로 치환되어 이메일 패턴(\S+@\S+)이 깨지므로 이메일이 올바르게 마스킹되지 않는다. (b)에서 extract_hashtags로 원본에서 해시태그를 추출하고 (c)에서 analyze_posts로 전체 게시물의 통계를 딕셔너리로 집계한다.

생성형 AI 활용 과정 링크: https://claude.ai/share/26f95c0b-b88e-4ecc-8cfc-0239b077dba0